# 73 · 수정 조회 후 카드 목록만 전달하는 순차 검색 비교

70번의 실제 검색 결과를 재사용해 U·기존 순차·수정 순차를 비교합니다. 표준 카드 순위 지표가 주 검색 지표이며, 정답은 세 방식에서 나온 후보 합집합을 원문으로 확인한 범위입니다. 같은 질문 3반복은 독립 질문이 아닙니다. 운영 코드·원본과 70~72 결과는 읽기 전용입니다. [실행 계획](notes/73_repaired_card_scope_plan.md).

In [1]:
from pathlib import Path
import collections, hashlib, importlib.util, json, math, statistics
import fitz

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
BASE = ROOT / 'notebooks/data/73_repaired_card_scope'
OLD = ROOT / 'notebooks/data/70_cap_context_repair'
EVIDENCE = ROOT / 'notebooks/data/71_pre_generation_evidence_audit'
BASE.mkdir(parents=True, exist_ok=True)
def read(path): return json.loads(Path(path).read_text())
def write(name, value): (BASE/name).write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n')
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
spec=importlib.util.spec_from_file_location('ranking73',ROOT/'notebooks/scripts/73_card_ranking.py')
ranking=importlib.util.module_from_spec(spec);spec.loader.exec_module(ranking)
print('73번 입력 경로 확인:', BASE.relative_to(ROOT))

73번 입력 경로 확인: notebooks/data/73_repaired_card_scope


## 0. DB3와 DB5의 실제 차이

70번의 기존 순차 DB3와 수정 순차 DB5를 읽기 전용으로 비교합니다. DB4에서 보완한 실적 파생값과 DB5에서 복구한 한도 문맥을 구분합니다.

In [2]:
import sqlite3
paths=[ROOT/'notebooks/data/60_structured_retrieval_repair/db_v3/structured.sqlite', OLD/'db_v5/structured.sqlite']
snapshots=[]
for path in paths:
    con=sqlite3.connect(path.resolve().as_uri()+'?mode=ro&immutable=1',uri=True);con.row_factory=sqlite3.Row
    snapshots.append({row['fact_id']:dict(row) for row in con.execute('select * from facts')});con.close()
assert len(snapshots[0])==len(snapshots[1])==3403 and snapshots[0].keys()==snapshots[1].keys()
changes=[];counts=collections.Counter()
for fact_id,a in snapshots[0].items():
    b=snapshots[1][fact_id];columns=[col for col in a if a[col]!=b[col]]
    if columns:
        counts.update(columns);changes.append(dict(fact_id=fact_id,card_key=a['document_id'],columns=columns,
            spend_before=a['spend_threshold_krw'],spend_after=b['spend_threshold_krw'],
            cap_raw_before=a['cap_raw'],cap_raw_after=b['cap_raw'],cap_state_before=a['cap_state'],cap_state_after=b['cap_state']))
assert len(changes)==169 and counts['spend_threshold_krw']==166 and counts['cap_raw']==3
write('db_difference.json',dict(db3_path=str(paths[0].relative_to(ROOT)),db5_path=str(paths[1].relative_to(ROOT)),
    db3_sha256=sha(paths[0]),db5_sha256=sha(paths[1]),total_fact_rows=3403,changed_fact_rows=169,columns=dict(counts),records=changes,
    explanation='166행은 DB4에서 전월실적 수치 파생값 보완; 3행은 DB5에서 할인 전 이용금액 상한 문맥 보완. 조회 명사 후보 규칙 변경은 DB 컬럼 차이와 별도.'))
print('전체3,403행 중 SQL 컬럼 차이:',len(changes),'행; 전월실적 수치:',counts['spend_threshold_krw'],'행, 이용금액 상한:',counts['cap_raw'],'행')

전체3,403행 중 SQL 컬럼 차이: 169 행; 전월실적 수치: 166 행, 이용금액 상한: 3 행


## 1. 원문 판정과 공통 후보 집합

71번의 확인된 카드 경로와 추가로 PDF를 직접 검토한 후보를 합칩니다. 원문 인용·소유 PDF·해시·쪽수를 확인합니다. 이미지 전사는 실제 렌더의 동일성을 확인하며, 의미 판정은 수동 검토입니다.

In [3]:
old=read(OLD/'retrieval_results.json')['records']
reviews=read(EVIDENCE/'input_reviews.json')['records']
contracts={x['query_id']:x for x in read(EVIDENCE/'question_contract.json')['records']}
source_seal=read(EVIDENCE/'input_review_seal.json')
assert sha(EVIDENCE/'input_reviews.json')==source_seal['reviews_raw_sha256']
assert sha(EVIDENCE/'question_contract.json')==source_seal['question_contract_raw_sha256']
for p,h in source_seal['files_sha256'].items(): assert sha(ROOT/p)==h,p
qids={'C01','C02','C03','C05','C09','C10','C11','C12','C13','C14','C16','C18'}
assert len(qids)==12 and all(contracts[q]['path_kind']=='same_card' for q in qids)
pool=collections.defaultdict(set); known=collections.defaultdict(set)
for row in old:
    if row['query_id'] in qids and row['repeat'] in (1,2):
        pool[row['query_id']].update(c['card_key'] for c in row['result']['cards'])
for review in reviews:
    if review['query_id'] not in qids: continue
    for path in review['paths']:
        if len(path['cards'])==1:
            known[review['query_id'],path['cards'][0]].add(path['eligibility'])
labels={q:{card:(True if 'verified' in known[q,card] else None) for card in cards}
        for q,cards in pool.items()}
need={(q,c) for q,cards in pool.items() for c in cards if 'verified' not in known[q,c]}
source=read(BASE/'additional_source_judgments.json')
additional=[r for r in source['records'] if (r['query_id'],r['card_key']) in need]
assert {(r['query_id'],r['card_key']) for r in additional}==need
questions={r['query_id']:r['question'] for r in read(OLD/'questions.json')['queries']}
checks=collections.Counter(); rendered={}
for row in additional:
    q,card=row['query_id'],row['card_key']
    assert row['question']==questions[q] and row['reasoning']
    assert row['source_pdf']['path']=='data/raw/'+card+'.pdf'
    pdf=ROOT/row['source_pdf']['path']; assert sha(pdf)==row['source_pdf']['sha256']
    assert row['relevance'] is None or type(row['relevance']) is bool
    with fitz.open(pdf) as doc:
        assert len(doc)==row['source_pdf']['page_count']
        for cite in row['source_quotes']:
            assert cite['pdf_path']==row['source_pdf']['path'] and cite['pdf_sha256']==row['source_pdf']['sha256']
            page=doc[cite['page']-1]
            if cite['quote_method']=='native_pdf_text':
                text=page.get_text(); assert hashlib.sha256(text.encode()).hexdigest()==cite['page_text_sha256']
                assert text[cite['start_char']:cite['end_char']]==cite['quote'];checks['native']+=1
            elif cite['quote_method']=='direct_pdf_visual_transcription':
                recipe=cite['render_recipe']; pix=page.get_pixmap(matrix=fitz.Matrix(recipe['scale'],recipe['scale']),alpha=recipe['alpha'])
                assert [pix.width,pix.height]==[recipe['width'],recipe['height']]
                assert hashlib.sha256(pix.samples).hexdigest()==recipe['rendered_pixels_sha256'] and cite['visually_reviewed']
                checks['visual_render']+=1
            else: raise ValueError('unknown quote method')
    labels[q][card]=row['relevance']
assert checks['native']>0 and checks['visual_render']>0
write('query_gold.json',dict(status='source_reviewed_judged_union_not_full_corpus',labels=labels,
    source_71_sha256=sha(EVIDENCE/'input_reviews.json'),additional_sha256=sha(BASE/'additional_source_judgments.json'),
    evaluation_queries=sorted(qids),excluded=['선택하지 않은 12개 질문은 사용자 요청에 따른 비용 절감 범위 축소'],
    caveat='70 결과를 본 뒤 원문을 검토한 개발 후보 집합. 106카드 전체 정답셋·독립 맹검 아님. null은 미확인.'))
write('source_verification.json',dict(additional_pairs=len(additional),source_pdf_citations=dict(checks),
    counts=dict(collections.Counter('unknown' if r['relevance'] is None else str(r['relevance']).lower() for r in additional)),
    additional_source_sha256=sha(BASE/'additional_source_judgments.json'),source71_seal_verified=True,
    note='원문 SHA·native 위치·렌더 재현을 자동 검증. 시각 전사의 의미와 카드 관계는 수동 검토.'))
print('공통 후보 수:', sum(len(x) for x in labels.values()), '질문 수:',len(labels), '추가 원문 판정:',dict(checks))
print('미확인 후보가 있는 질문:',[q for q,v in labels.items() if None in v.values()])

공통 후보 수: 56 질문 수: 12 추가 원문 판정: {'native': 54, 'visual_render': 13}
미확인 후보가 있는 질문: ['C11', 'C14']


## 2. 표준 검색 순위 지표와 구조화 조회 경로

카드 Hit@3·MRR@3은 미확인을 성공/실패로 가정한 범위를 함께 기록합니다. Recall@3·nDCG@3은 **세 검색 방식의 후보 합집합에서 원문 판정이 완료된 질문**에 한해 계산합니다. 전체 106카드 회수율이 아닙니다. 상위 5개를 반환하지 않아 @5는 보고하지 않습니다.

In [4]:
profiles=('card_page_section_benefit','parent_child_bundle');modes=('U','SEQ65','SEQ_REPAIRED')
rank_rows=[]
for row in old:
    q=row['query_id']
    if q not in qids or row['repeat'] not in (1,2): continue
    cards=row['result']['cards']; order=[c['card_key'] for c in cards]
    assert all(c['rank']==i for i,c in enumerate(cards,1))
    rank_rows.append(dict(identity=row['identity'],query_id=q,profile=row['profile'],mode=row['mode'],repeat=row['repeat'],
        cards=order,**ranking.score(order,labels[q],3)))
assert len(rank_rows)==12*2*3*2
summary=[]; by_question=[]
metrics=('hit_lower','hit_upper','mrr_lower','mrr_upper','recall_judged_pool','ndcg_judged_pool')
for profile in profiles:
    for mode in modes:
        selected=[r for r in rank_rows if r['profile']==profile and r['mode']==mode]
        row=dict(profile=profile,mode=mode,questions=12,repeats_per_question=2)
        for metric in metrics:
            values=[]
            for q in sorted(qids):
                group=[r[metric] for r in selected if r['query_id']==q]
                assert len(group)==2
                if all(x is not None for x in group): values.append(statistics.mean(group))
            row[metric]=statistics.mean(values) if values else None; row[metric+'_questions']=len(values)
        summary.append(row)
        for q in sorted(qids):
            group=[r for r in selected if r['query_id']==q]
            by_question.append(dict(profile=profile,mode=mode,query_id=q,
                **{key:(statistics.mean([r[key] for r in group]) if all(r[key] is not None for r in group) else None) for key in metrics}))
assert len(summary)==6 and len(by_question)==72
write('ranking_results.json',dict(records=rank_rows,unit='question/card',k=3))
write('ranking_summary.json',dict(records=summary,per_question=by_question,source_scope='70번 저장된 세 방식 후보 합집합',
    relevance='71 확인 경로 + 73 PDF 판정; 모든 106개 카드를 전수 판정하지 않음',
    hit_mrr='미확인 후보를 각각 부적합/적합으로 본 하한·상한',
    recall_ndcg='질문별 후보 합집합 전체가 판정된 질문만, 합집합 내부 정답 기준',
    denominator='12개 단일카드 질문×2반복; 미확인 질문은 Recall/nDCG에서 모든 방식 공통 제외',
    user_requested_at5='실제 최종 카드 상위3개만 저장돼 @5를 계산하지 않음'))
for s in summary: print(s['profile'],s['mode'], 'Hit',round(s['hit_lower'],4),round(s['hit_upper'],4),
 'MRR',round(s['mrr_lower'],4),round(s['mrr_upper'],4),'Recall(pool)',round(s['recall_judged_pool'],4),
 'nDCG(pool)',round(s['ndcg_judged_pool'],4),'유효질문',s['recall_judged_pool_questions'])
routing=[r for r in read(OLD/'routing_comparisons.json')['records'] if r['query_id'] in qids and r['repeat'] in (1,2)]
assert len(routing)==24
counts=collections.Counter((r['SEQ65']['route'],r['SEQ_REPAIRED']['route']) for r in routing)
changed=[]
for r in routing:
    old_scope=r['SEQ65']['scope']; new_scope=r['SEQ_REPAIRED']['scope']
    if old_scope!=new_scope:changed.append(dict(query_id=r['query_id'],repeat=r['repeat'],old_count=len(old_scope),new_count=len(new_scope),
        removed=sorted(set(old_scope)-set(new_scope)),added=sorted(set(new_scope)-set(old_scope))))
write('routing_diagnostics.json',dict(route_transitions=[dict(old=a,new=b,count=n) for (a,b),n in counts.items()],
    scope_changed=changed,total=24,reason='구조화 결과·후보 변화는 질문 정답률이 아님'))
print('라우팅 경로 변화:',dict(counts),'카드 범위 변경:',len(changed),'/24')

card_page_section_benefit U Hit 0.9167 1 MRR 0.9167 0.9583 Recall(pool) 0.6831 nDCG(pool) 0.9173 유효질문 10
card_page_section_benefit SEQ65 Hit 1 1 MRR 1.0 1.0 Recall(pool) 0.7081 nDCG(pool) 0.9531 유효질문 10
card_page_section_benefit SEQ_REPAIRED Hit 1 1 MRR 1.0 1.0 Recall(pool) 0.7081 nDCG(pool) 0.9531 유효질문 10
parent_child_bundle U Hit 1 1 MRR 1.0 1.0 Recall(pool) 0.6631 nDCG(pool) 0.8877 유효질문 10
parent_child_bundle SEQ65 Hit 1 1 MRR 1.0 1.0 Recall(pool) 0.7081 nDCG(pool) 0.9531 유효질문 10
parent_child_bundle SEQ_REPAIRED Hit 1 1 MRR 1.0 1.0 Recall(pool) 0.7081 nDCG(pool) 0.9531 유효질문 10
라우팅 경로 변화: {('SEQ', 'SEQ'): 21, ('U', 'U'): 3} 카드 범위 변경: 2 /24


## 3. 답변 생성과 최종 검토

이 셀은 유료 API 호출을 자동으로 실행하지 않습니다. 비용 예약, 입력·표준 지표·독립 코드 검토를 마친 후 73번 실행기로 직렬 실행합니다. 실행기의 중단 재개와 원 응답 저장 때문에 보조 파일을 사용합니다. 사람이 답변을 원문·실제 입력과 대조한 뒤 별도 산출물과 HTML에 반영합니다.

In [5]:
reservation=read(BASE/'cost_reservation.json')
print('새 답변:',reservation['logical_answers'],'최대 물리 호출:',reservation['maximum_physical_requests'])
print('기존 누적 차감:',reservation['starting_budget_debit_usd'],'최대 새 예약:',reservation['maximum_new_usd'])
print('유료 실행은 독립 검토와 pre_paid_acceptance.json 확인 후 별도 실행기에서 수행합니다.')

새 답변: 48 최대 물리 호출: 96
기존 누적 차감: 2.772806085 최대 새 예약: 0.591936000
유료 실행은 독립 검토와 pre_paid_acceptance.json 확인 후 별도 실행기에서 수행합니다.
